# Reproduce the real-data Metrics examples

This optional lab reads the original UCI files and reproduces the model outputs used in the interactive lessons.

- **Classification** · Standardize 57 email features and fit logistic regression on training data. Score a separate test fold and evaluate Spam decisions at three example cutoffs.
- **Regression** · Fit historical rental count averages by hour and working-day status. Forecast later hours and compare absolute and squared errors with a training-mean baseline.

Run the cells in sequence. Set `UCI_SPAMBASE_FILE` and `UCI_BIKESHARE_ARCHIVE` in your environment to use local source copies. Otherwise the notebook fetches the official inputs. The published Spambase scores use scikit-learn 1.6.1 and are rounded to four decimal places. Small solver or version differences near those rounding boundaries can change a score.

In [ ]:
import io
import math
import os
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SPAM_SOURCE = 'https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data'
BIKE_SOURCE = 'https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip'
SPLIT_SEED = 37
ROC_DECIMALS = 4
BIKE_TEST_START = '2012-11-01'

def read_source(local_env, url):
    local_copy = os.environ.get(local_env)
    if local_copy:
        return Path(local_copy).read_bytes()
    with urlopen(url, timeout=30) as response:
        return response.read()

print('scikit-learn', sklearn.__version__)

## 1. Keep identical email feature vectors together

Spambase supplies 57 numerical features and a recorded Spam label. The features include word and character frequencies and capital-letter run statistics. The final column says whether the record is Spam (1) or Legitimate (0).

We take the first fold from a seeded stratified group split. Stratifying aims to preserve class proportions. Grouping prevents identical feature rows from appearing in both training and test data. This is the same split used in the Bagging implementation. The scaler and classifier fit only on the training portion.

In [ ]:
spam_records = pd.read_csv(io.BytesIO(read_source('UCI_SPAMBASE_FILE', SPAM_SOURCE)), header=None)
spam_X, spam_y = spam_records.iloc[:, :-1], spam_records.iloc[:, -1]
assert spam_X.shape == (4601, 57)
groups = spam_X.groupby(spam_X.columns.tolist(), sort=False, dropna=False).ngroup()
split = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SPLIT_SEED)
spam_train, spam_test = next(split.split(spam_X, spam_y, groups))
assert set(groups.iloc[spam_train]).isdisjoint(groups.iloc[spam_test])

spam_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1500))
spam_model.fit(spam_X.iloc[spam_train], spam_y.iloc[spam_train])
spam_class = list(spam_model.classes_).index(1)
spam_scores = np.round(spam_model.predict_proba(spam_X.iloc[spam_test])[:, spam_class], ROC_DECIMALS)
spam_frame = pd.DataFrame({
    'row_number': spam_test + 1,
    'score': spam_scores,
    'actual': spam_y.iloc[spam_test].to_numpy(),
})
print(f'Fitted on {len(spam_train):,} emails and scored {len(spam_test):,} held-out emails.')

## 2. Use the same scored emails at every cutoff

Use scores to make predictions. A score at or above the cutoff goes to the spam folder. Use the recorded labels afterwards to count true positives, false negatives, false positives and true negatives. The numbers compare example policies on fixed data. In a production evaluation, choose the cutoff on validation data and lock it before measuring on the final test set.

One point on ROC comes from one cutoff. AUC uses the underlying rank of the scores. Treat the fitted probabilities as ranking scores until you have tested their calibration for the intended mail population.

In [ ]:
def spam_metrics(frame, cutoff):
    actual = frame['actual']
    predicted = (frame['score'] >= cutoff).astype(int)
    tp, fn, fp, tn = (int(value) for value in confusion_matrix(actual, predicted, labels=[1, 0]).ravel())
    return {
        'tp': tp, 'fn': fn, 'fp': fp, 'tn': tn,
        'precision': tp / (tp + fp) if tp + fp else math.nan,
        'recall': tp / (tp + fn),
        'fpr': fp / (fp + tn),
        'accuracy': (tp + tn) / len(actual),
    }

thresholds = (0.20, 0.50, 0.75)
full_spam_metrics = pd.DataFrame({t: spam_metrics(spam_frame, t) for t in thresholds}).T
full_spam_auc = roc_auc_score(spam_frame['actual'], spam_frame['score'])
print(f'Full test fold · {int(spam_frame["actual"].sum())} Spam · ROC AUC {full_spam_auc:.6f}')
print(full_spam_metrics.round(4))

### Recreate the 20 teaching emails

Sort the held-out emails by score within each recorded class. Pick 10 approximately evenly spaced ranks from each class. Use the original source row number to break equal-score sort ties. These rules give a manageable classroom example with actual rows and model scores. The selected 20 have a different class composition and score distribution from the full test fold.

In [ ]:
selected = []
for label in (0, 1):
    ranked = spam_frame[spam_frame['actual'] == label].sort_values(['score', 'row_number'])
    positions = np.rint(np.linspace(0, len(ranked) - 1, 10)).astype(int)
    selected.append(ranked.iloc[positions])
spam_examples = pd.concat(selected).sort_values(['score', 'row_number'], ascending=[False, True])
spam_ids = [f'E{number}' for number in spam_examples['row_number']]
spam_examples = spam_examples.assign(id=spam_ids)
teaching_spam_metrics = spam_metrics(spam_examples, 0.50)
teaching_spam_auc = roc_auc_score(spam_examples['actual'], spam_examples['score'])
print(spam_examples.set_index('id')[['score', 'actual']].to_string(float_format=lambda value: f'{value:.4f}'))
print(f'Teaching sample at 0.50 · {teaching_spam_metrics} · AUC {teaching_spam_auc:.3f}')

## 3. Split rental hours by time and fit calendar averages

The UCI archive contains a file of hourly Capital Bikeshare counts. Use the rows dated before November 1, 2012 to fit the model. Test on the available hourly rows from November 1 and later. A chronological split matches the goal of forecasting a future hour using past counts. The rule can know the hour and whether the date is a working day ahead of time. Actual counts become available after that hour.

The raw `cnt` is the total count of rentals. The `casual` and `registered` counts are already components of that observed total and are excluded from the forecast. The fitted calendar model uses only the historical average with the same hour and working-day indicator. The constant baseline is the average of all historical training counts. We round both forecasts to whole rentals.

In [ ]:
with ZipFile(io.BytesIO(read_source('UCI_BIKESHARE_ARCHIVE', BIKE_SOURCE))) as archive:
    with archive.open('hour.csv') as raw:
        bike_records = pd.read_csv(raw, usecols=['dteday', 'hr', 'workingday', 'cnt'])
bike_records = bike_records.sort_values(['dteday', 'hr']).reset_index(drop=True)
assert len(bike_records) == 17379
assert not bike_records.duplicated(['dteday', 'hr']).any()
bike_train = bike_records[bike_records['dteday'] < BIKE_TEST_START]
bike_test = bike_records[bike_records['dteday'] >= BIKE_TEST_START].copy()

by_calendar = bike_train.groupby(['hr', 'workingday'])['cnt'].agg(count='size', total='sum')
by_calendar['predicted'] = np.rint(by_calendar['total'] / by_calendar['count']).astype(int)
bike_test = bike_test.join(by_calendar, on=['hr', 'workingday'])
assert bike_test['predicted'].notna().all()
training_baseline = round(bike_train['cnt'].mean())
print(f'Used {len(bike_train):,} training hours, before {BIKE_TEST_START}, and {len(bike_test):,} test hours.')
print(f'Rounded training mean baseline: {training_baseline} rentals.')

## 4. Evaluate each rental forecast after the recorded count arrives

Use the same held-out rows for the calendar model and for the training-mean baseline. MAE averages the absolute misses. MSE averages squared misses. RMSE returns the result to rentals. R² compares squared errors with the variation around the mean of the recorded evaluation counts. That evaluation mean is a reference in the metric, while the two deployable forecasts depend on historical training data.

The five teaching rows are the first day’s hours from 07:00 to 11:00. Use the training group totals to verify forecasts before checking their actual counts.

In [ ]:
def bike_metrics(actual, predicted):
    residuals = actual - predicted
    sse = float((residuals ** 2).sum())
    mse = sse / len(actual)
    actual_mean = actual.mean()
    sst = math.fsum((value - actual_mean) ** 2 for value in actual)
    return {
        'mae': float(residuals.abs().mean()),
        'mse': mse,
        'rmse': math.sqrt(mse),
        'r2': 1 - sse / sst if sst else math.nan,
    }

bike_examples = bike_test[(bike_test['dteday'] == BIKE_TEST_START) & bike_test['hr'].between(7, 11)]
teaching_bike_metrics = bike_metrics(bike_examples['cnt'], bike_examples['predicted'])
full_bike_metrics = {
    'calendar_model': bike_metrics(bike_test['cnt'], bike_test['predicted']),
    'training_mean': bike_metrics(bike_test['cnt'], training_baseline),
}
print(bike_examples[['dteday', 'hr', 'workingday', 'count', 'total', 'predicted', 'cnt']].to_string(index=False))
print(f'Teaching five-hour metrics · {teaching_bike_metrics}')
print(pd.DataFrame(full_bike_metrics).T.round(6))

## Take the results back to the lesson

The scores and forecasts come from UCI data measured in their original contexts. The folder decision is a lesson policy. The threshold, sample size, group split, and calendar rule are documented choices. The reported performance applies to these held-out records. A current mail filter needs a recent evaluation set. A current bike forecast needs a new chronological evaluation that reflects the system you want to plan for.

Open [Classification Metrics](https://sharanharsoor.github.io/ml-learning-lab/classification-regression-metrics/#classification) or [Regression Metrics](https://sharanharsoor.github.io/ml-learning-lab/classification-regression-metrics/#regression) to explore the fixed teaching examples.

## Sources

- [Hopkins, Reeber, Forman, and Suermondt (1999), Spambase, UCI Machine Learning Repository](https://doi.org/10.24432/C53G6X)
- [Fanaee-T (2013), Bike Sharing, UCI Machine Learning Repository](https://doi.org/10.24432/C5W894)
- Both datasets are licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).